# Isharah-500 SI experiment comparison

This notebook loads actual saved training histories and the corrected SI dev error analysis. It compares the SI pilot, hand-centric five-epoch run, hand-centric ten-epoch resume, and hand-centric fifteen-epoch resume. It never reads SI test labels and never trains.

In [ ]:
from pathlib import Path
import json
import os
def resolve_project_root():
    configured = os.environ.get("ISHARAH_PROJECT_ROOT")
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "isharah500" / "src").is_dir():
            return candidate
    raise RuntimeError("Set ISHARAH_PROJECT_ROOT to the repository worktree root.")

PROJECT_ROOT = resolve_project_root()
ISHARAH_ROOT = PROJECT_ROOT / "isharah500"
CHECKPOINT_ROOT = ISHARAH_ROOT / "checkpoints"
REPORT_ROOT = ISHARAH_ROOT / "reports"
RUNS = {"si_pilot_5ep": CHECKPOINT_ROOT / "si_pilot" / "isharah500_si_training_history.json", "hand_centric_5ep": CHECKPOINT_ROOT / "si_hand_centric_5ep" / "isharah500_si_training_history.json", "hand_centric_resume_10ep": CHECKPOINT_ROOT / "si_hand_centric_resume_10ep" / "isharah500_si_training_history.json", "hand_centric_resume_15ep": CHECKPOINT_ROOT / "si_hand_centric_resume_15ep" / "isharah500_si_training_history.json"}
CORRECTED_REPORT = REPORT_ROOT / "si_error_analysis_epoch15_corrected.json"
def load_history(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    history = json.loads(path.read_text(encoding="utf-8"))
    required_top = {"epoch", "train", "validation"}
    required_metrics = {"ctc_loss", "gloss_error_rate", "gloss_exact_match"}
    if not isinstance(history, list) or not history:
        raise ValueError(f"Training history must be a non-empty list: {path}")
    for row in history:
        if not required_top.issubset(row) or not required_metrics.issubset(row["train"]) or not required_metrics.issubset(row["validation"]):
            raise ValueError(f"Unexpected training history schema: {path}")
    return history

histories = {name: load_history(path) for name, path in RUNS.items()}
corrected = json.loads(CORRECTED_REPORT.read_text(encoding="utf-8"))
required_report = {"protocol", "split", "sample_count", "analysis"}
required_aggregate = {"gloss_error_rate", "exact_match", "substitutions", "deletions", "insertions"}
if not required_report.issubset(corrected) or not required_aggregate.issubset(corrected["analysis"]["aggregate"]):
    raise ValueError(f"Unexpected corrected report schema: {CORRECTED_REPORT}")
assert corrected["protocol"] == "SI" and corrected["split"] == "dev"
print({name: len(history) for name, history in histories.items()})
print("Corrected report source:", CORRECTED_REPORT)

## Metric provenance

Training-history CTC loss and GER values are read from each run's saved `isharah_si_training_history.json`. The corrected epoch 15 Exact Match, GER, edit counts, and length statistics are read from `si_error_analysis_epoch15_corrected.json`, which evaluates all 500 SI dev samples.

The epoch 15 training history contains Exact Match `0.28`; that value is historical and incorrect because the pre-refactor aggregation mishandled exact-match accumulation across batches. The canonical corrected value is `0.07` (35/500). It is not silently substituted into the historical history file.

In [ ]:
def final_history_row(history):
    return history[-1]

comparison = []
for name, history in histories.items():
    row = final_history_row(history)
    historical_exact = row["validation"]["gloss_exact_match"]
    record = {
        "experiment": name,
        "epochs": row["epoch"],
        "source": str(RUNS[name]),
        "validation_ctc_loss": row["validation"]["ctc_loss"],
        "validation_ger": row["validation"]["gloss_error_rate"],
        "exact_match": historical_exact,
    }
    if name == "hand_centric_resume_15ep":
        record["exact_match"] = None
        record["historical_incorrect_exact_match"] = historical_exact
    comparison.append(record)

corrected_aggregate = corrected["analysis"]["aggregate"]
comparison.append({
    "experiment": "hand_centric_resume_15ep_corrected",
    "epochs": 15,
    "source": str(CORRECTED_REPORT),
    "validation_ctc_loss": final_history_row(histories["hand_centric_resume_15ep"])["validation"]["ctc_loss"],
    "validation_ger": corrected_aggregate["gloss_error_rate"],
    "exact_match": corrected_aggregate["exact_match"],
})
comparison

## Training curves

The following visualization uses only values loaded from saved histories. It does not recompute or invent metrics.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for name, history in histories.items():
    epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["train"]["ctc_loss"] for row in history], marker="o", label=name)
    axes[1].plot(epochs, [row["validation"]["ctc_loss"] for row in history], marker="o", label=name)
    axes[2].plot(epochs, [row["validation"]["gloss_error_rate"] for row in history], marker="o", label=name)
axes[0].set_title("Train CTC loss")
axes[1].set_title("SI dev CTC loss")
axes[2].set_title("SI dev GER")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.grid(alpha=0.3)
axes[0].legend(fontsize="small")
plt.tight_layout()
plt.show()

## Actual final comparison

This chart uses the final saved row from each history. The corrected epoch 15 Exact Match is sourced from the corrected SI dev report rather than the historical training-history field.

In [ ]:
valid_exact_comparison = [row for row in comparison if row["exact_match"] is not None]
labels = [row["experiment"] for row in valid_exact_comparison]
gers = [row["validation_ger"] for row in comparison]
exact = [row["exact_match"] for row in valid_exact_comparison]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar([row["experiment"] for row in comparison], gers)
axes[0].set_title("SI dev GER by source")
axes[0].tick_params(axis="x", rotation=35)
axes[1].bar(labels, exact)
axes[1].set_title("SI dev Exact Match; corrected source only")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()